### Clone the repo

In [1]:
import os

%cd /content/

if not os.path.exists("sssumo"):
    !git clone https://github.com/dolphin-in-a-coma/sssumo.git
    print("Repository cloned successfully.")
else:
    print("Repository already exists.")

%cd sssumo/src

# Pulling data
!git lfs install
!git lfs pull

/content
Cloning into 'sssumo'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 44 (delta 19), reused 35 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 12.51 MiB | 17.46 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Repository cloned successfully.
/content/sssumo

--- Top Level Directory ---
checkpoints/  configs/	LICENSE  README.md  src/


In [2]:
!pip install torchviz
!pip install fastkde


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.0 MB/s eta 0:00:00


### Specify model and data

In [17]:
root_dir = '/content/sssumo/'

nn_config_dir = os.path.join(root_dir, 'configs')
nn_checkpoint_dir = os.path.join(root_dir, 'checkpoints')
datasets_dir = os.path.join(root_dir, 'data')

nn_config_name = 'config-0426-ModGaussian_no_reconstruction.yaml'
nn_checkpoint_number = 24
nn_checkpoint_name = f'{nn_config_name.split(".")[0]}_{nn_checkpoint_number}.pth'

nn_config_path = os.path.join(root_dir, nn_config_dir, nn_config_name)
nn_checkpoint_path = os.path.join(root_dir, nn_checkpoint_dir, nn_checkpoint_name)


data_source = 'synthetic'

# root_dir =

# datasets_dir = f'{root_dir}/data/'
dataset2path = {
     # 'crank1d':
    'steering': os.path.join(datasets_dir, 'steering_tangential_velocity_data.csv'),
    'crank': os.path.join(datasets_dir, 'crank_tangential_velocity_data.csv'),
    'Fitts': os.path.join(datasets_dir, 'Fitts_tangential_velocity_data.csv'),
    'whacamole': os.path.join(datasets_dir, 'whacamole_tangential_velocity_data.csv'),
    'object_moving': os.path.join(datasets_dir, 'object_moving_tangential_velocity_data.csv'),
    'pointing': os.path.join(datasets_dir, 'pointing_tangential_velocity_data.csv'),
    'tablet_writing': os.path.join(datasets_dir, 'tablet_writing_tangential_velocity_data.csv'),
}

In [14]:
from data import SyntheticDataset, OrganicDataset, CombinedSyntheticDataset
from models import TDNNDetector, STEContinuousReconstructor, STEBinarizer
from utils import onset_prediction_metrics_on_masks, Config, evaluate_on_organic_data, evaluate_on_synthetic_data, \
    calculate_reconstruction_metrics, calculate_supervised_metrics, \
    evaluate_organic_trials_for_bootstrap, hierarchical_bootstrap_metrics

In [16]:
config = Config(nn_config_path, root_dir=root_dir)

In [4]:
# !git lfs pull

Updated git hooks.
Git LFS initialized.


In [18]:

config = Config(nn_config_path, root_dir=root_dir)

# if MODEL_TYPE != 'TDNN':
#     config.device = torch.device('cpu')

# config.experiment_name = CONFIG_PATH.split('/')[-1].replace('.yaml', '') + '_for_test'
config.start_with_weights = nn_checkpoint_path
# config.proportions = proportions

# Some basic dataset parameters, can be changed further down
config.total_duration_distribution = 1000
config.batch_size = 512 # does it affect the running?
config.num_samples = 1
config.refractory_distribution = [0, 1.5] # this is going to be used for the synthetic dataset

# os.makedirs(os.path.dirname(config.log_file), exist_ok=True)
# writer = SummaryWriter(log_dir=config.log_dir)
print(config.experiment_name)

# Load the model

model = TDNNDetector(
        batchnorm=config.batchnorm,
        dilations=config.dilations,
        channels=config.channels,
        kernel_sizes=config.kernel_sizes,
        num_layers=config.num_layers,
        dropout_rate=config.dropout_rate,
    ).to(config.device, config.dtype)

model.eval()


weights_file = config.start_with_weights
weights_dir = '/'.join(config.weights_file.split('/')[:-1])
weights_file = os.path.join(weights_dir, weights_file)


basic_dataset = SyntheticDataset(**config.get_dataset_parameters())

reconstructor = basic_dataset.reconstruction_model

train_noise_condition = config.stat_snr_distribution


0426-ModGaussian_no_reconstruction


/usr/local/lib/python3.12/dist-packages/torch/distributions/distribution.py:62: UserWarning: <class 'data.ModulatedGaussian'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


In [31]:
x_noisy, x_clean, y_true = basic_dataset[0]

In [27]:
y_pred = model(x_noisy)

tensor([[[ 0.5052,  0.5054,  0.5056,  ...,  0.5041,  0.5043,  0.5050],
         [-0.0151, -0.0153, -0.0152,  ..., -0.0157, -0.0162, -0.0156],
         [ 0.0157,  0.0165,  0.0175,  ...,  0.0176,  0.0169,  0.0183]],

        [[ 0.5052,  0.5054,  0.5056,  ...,  0.5043,  0.5044,  0.5051],
         [-0.0152, -0.0155, -0.0153,  ..., -0.0162, -0.0165, -0.0157],
         [ 0.0157,  0.0166,  0.0176,  ...,  0.0177,  0.0171,  0.0185]],

        [[ 0.5052,  0.5054,  0.5056,  ...,  0.5044,  0.5044,  0.5051],
         [-0.0152, -0.0154, -0.0152,  ..., -0.0166, -0.0164, -0.0158],
         [ 0.0157,  0.0166,  0.0175,  ...,  0.0178,  0.0171,  0.0186]],

        ...,

        [[ 0.5052,  0.5055,  0.5057,  ...,  0.5041,  0.5042,  0.5050],
         [-0.0152, -0.0155, -0.0153,  ..., -0.0163, -0.0164, -0.0159],
         [ 0.0157,  0.0167,  0.0175,  ...,  0.0176,  0.0169,  0.0183]],

        [[ 0.5052,  0.5054,  0.5056,  ...,  0.5044,  0.5045,  0.5051],
         [-0.0151, -0.0155, -0.0153,  ..., -0.0164, -0.

In [34]:
x_noisy.shape

torch.Size([512, 1, 1000])

In [32]:
y_true.shape

torch.Size([512, 3, 1000])

In [ ]:
!git